# bwiki 查询智能ai agent
### 实现类似于在 QQ 群里 @ 一个机器人并发送“@robotname + 查询：角色名”，它就会自动去Bilibili百科检索该角色，并用简洁口语化的方式在群里回复关键信息。

## 1.准备工作
- 为了能从python程序中访问网络api，我们需要一个HTTP库，requests时python的一个库，用于发送HTTP请求。
- tavily-python是一个强大的ai搜索的api客户端，用于获取实时的网络搜索的结果，可以在官网注册后获取API
- openai时OpenAI官方提供的Python SDK，用于调用GPT等大语言模型服务

In [6]:
%pip install requests tavily-python openai python-dotenv
%pip install requests mwparserfromhell beautifulsoup4
%pip freeze> ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# 2.测试API的获取，并分析相应结构
https://wiki.biligame.com/blhx/api.php?action=query&prop=revisions&rvslots=*&rvprop=content&titles=%E4%BF%A1%E6%B5%93&format=json

In [1]:
import requests
from requests.exceptions import HTTPError, ConnectionError, Timeout
import mwparserfromhell
from urllib.parse import quote
import re
import json
import time


def get_blhx_wiki_data(name: str):
    """
    碧蓝海事局wiki统一查询工具
    内部API仅请求1次，无重试循环；自动解析舰娘图鉴/鱼雷图鉴/#invoke装备图鉴
    :param name: wiki页面标题
    :return: dict，解析后的图鉴数据；失败返回None
    """
    def clean_wiki_text(text: str) -> str:
        """清洗wiki标记"""
        text = re.sub(r"<br\s*/?>", "\n", text)
        text = re.sub(r"\[\[([^|]*\|)?([^\]]*)\]\]", r"\2", text)
        text = re.sub(r"{{黑幕\|([^}]*)}}", r"\1", text)
        text = re.sub(r"{{.*?}}", "", text)
        return text.strip()

    # ========== 网络请求：只请求一次，没有for重试循环 ==========
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    url = (
        f"https://wiki.biligame.com/blhx/api.php"
        f"?action=query&prop=revisions&rvslots=*&rvprop=content"
        f"&titles={quote(name)}&format=json"
    )
    try:
        resp = requests.get(url, timeout=15, headers=headers)
        if resp.status_code >= 400:
            raise HTTPError(f"Status {resp.status_code}")
        data = resp.json()
        pages = data["query"]["pages"]
        page = list(pages.values())[0]
        if "missing" in page:
            print(f"页面不存在：{name}")
            return None
        wikitext = page["revisions"][0]["slots"]["main"]["*"]
    except HTTPError as e:
        print(f"【HTTP错误】: {e}")
        return None
    except (ConnectionError, Timeout) as e:
        print(f"【网络/超时】: {e}")
        return None
    except Exception as e:
        print(f"【未知异常】: {e}")
        return None

    # ========== 解析模板，支持普通图鉴 和 #invoke lua图鉴 ==========
    wiki_code = mwparserfromhell.parse(wikitext)
    templates = wiki_code.filter_templates()
    for tpl in templates:
        tpl_name = str(tpl.name).strip()
        data_out = {"_template_name": tpl_name}

        # #invoke:装备图鉴 这类Lua模板，跳过第一个main参数
        if tpl_name.startswith("#invoke:") and tpl_name.endswith("图鉴"):
            for idx, param in enumerate(tpl.params):
                if idx == 0:
                    continue
                key = str(param.name).strip()
                val = clean_wiki_text(str(param.value).strip())
                if key:
                    data_out[key] = val
            print(json.dumps(data_out, ensure_ascii=False, indent=2))
            time.sleep(1.2)
            return data_out

        # 老式模板：舰娘图鉴、鱼雷图鉴
        if tpl_name.endswith("图鉴"):
            for param in tpl.params:
                key = str(param.name).strip()
                val = clean_wiki_text(str(param.value).strip())
                if key:
                    data_out[key] = val
            print(json.dumps(data_out, ensure_ascii=False, indent=2))
            time.sleep(1.2)
            return data_out

    print(f"页面[{name}]没有找到xxx图鉴模板")
    return None


if __name__ == "__main__":
    get_blhx_wiki_data("信浓")
    # get_blhx_wiki_data("F6F地狱猫")
    # get_blhx_wiki_data("五联装533mm鱼雷Mark IXT0")


{
  "_template_name": "舰娘图鉴",
  "分组": "",
  "特殊底色": "",
  "型号": "大和级战列舰改装航空母舰",
  "名称": "信浓",
  "和谐名": "鵗",
  "英文名": "IJN Shinano",
  "日文名": "信濃 しなの",
  "编号": "231",
  "类型": "航母",
  "初始星级": "★★★☆☆☆",
  "稀有度": "海上传奇",
  "阵营": "重樱",
  "掉落点": "",
  "活动掉落点": "",
  "其他获取途径": "蝶海梦花累计建造200次活动池后兑换获取\n复刻蝶海梦花累计建造200次活动池后兑换获取\n常驻UR兑换",
  "相关活动": "蝶海梦花",
  "耗时": "05:15:00",
  "营养价值": "",
  "退役收益": "金币9 燃油3 勋章30 特装原型500",
  "需强化炮击": "0",
  "强化每点炮击所需经验": "0",
  "需强化雷击": "0",
  "强化每点雷击所需经验": "0",
  "需强化航空": "83",
  "强化每点航空所需经验": "20",
  "需强化装填": "35",
  "强化每点装填所需经验": "40",
  "解锁图鉴科技点": "26",
  "突破至满星科技点": "52",
  "达到120级科技点": "39",
  "解锁图鉴属性加成": "轻航/正航 耐久+2",
  "达到120级属性加成": "轻航/正航 航空+1",
  "图鉴耐久": "S",
  "图鉴防空": "B",
  "图鉴机动": "B",
  "图鉴航空": "S",
  "图鉴雷击": "E",
  "图鉴炮击": "E",
  "装甲类型": "重型",
  "初始耐久": "1462",
  "初始装填": "46",
  "初始命中": "27",
  "初始炮击": "0",
  "初始雷击": "0",
  "初始机动": "11",
  "初始防空": "60",
  "初始航空": "84",
  "初始消耗": "8",
  "初始反潜": "0",
  "满级耐久": "8818",
  "满级装填": "128",
  "满级命中": "80",
  

## 导入qq-botpy库
实现将agent接入到qq机器人中

In [3]:
%pip install qq-botpy
%pip freeze > cd../cd../requirements.txt

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ϵͳ�Ҳ���ָ����·����
